# League AI Coach — deployment verification

This notebook proves the app is **deployed and running**, and that its Databricks
components produce **real, live output**. It was executed on **2026-09-24** against
workspace `8259563309010363` (profile `fevm-central1`); the cell outputs below are the
captured live results. Every cell uses the Databricks SDK / endpoints, so it is
re-runnable in that workspace.

Live app URL: https://league-ai-coach-8259563309010363.gcp.databricksapps.com


## 1. The app is deployed and RUNNING
Read the app's live status straight from the Apps API.


In [1]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
app = w.apps.get("league-ai-coach")
print("name:   ", app.name)
print("url:    ", app.url)
print("app:    ", app.app_status.state, "-", app.app_status.message)
print("compute:", app.compute_status.state, "-", app.compute_status.message)


name:    league-ai-coach
url:     https://league-ai-coach-8259563309010363.gcp.databricksapps.com
app:     RUNNING - App has status: App is running
compute: ACTIVE - App compute is running.


## 2. Benchmark data built by the Spark Declarative Pipeline
The weekly SDP rebuilds the tier/role benchmark medallion as materialized
views. All 7 tiers x 5 roles x 20 metrics = 700 rows in `gold_challenge_benchmarks`.


In [1]:
WAREHOUSE = "f3846cabca43f2d1"
def sql(q):
    r = w.statement_execution.execute_statement(warehouse_id=WAREHOUSE, wait_timeout="30s",
        statement=q)
    return r.result.data_array if r.result else []

for row in sql("SELECT tier, COUNT(*) AS rows FROM "
               "fevm_central1.league_ai_coach.gold_challenge_benchmarks "
               "GROUP BY tier ORDER BY tier"):
    print(row)


['BRONZE', '100']
['DIAMOND', '100']
['EMERALD', '100']
['GOLD', '100']
['IRON', '100']
['PLATINUM', '100']
['SILVER', '100']


In [1]:
for row in sql("SELECT metric, ROUND(gold_avg,2) AS gold_avg, sample_size FROM "
               "fevm_central1.league_ai_coach.gold_challenge_benchmarks "
               "WHERE tier='GOLD' AND team_position='BOTTOM' AND metric IN "
               "('laneMinionsFirst10Minutes','kda','goldPerMinute') ORDER BY metric"):
    print(row)


['goldPerMinute', '466.89', '16']
['kda', '2.36', '16']
['laneMinionsFirst10Minutes', '46.31', '16']


## 3. Genie Assistant via the Multi-Agent Supervisor
The in-app chatbot calls a Multi-Agent Supervisor that routes to a per-table
Genie agent. Here it routes a benchmark question to the `gold_challenge_benchmarks`
agent and answers with the real value (466.89 — matches cell 2).


In [1]:
resp = w.api_client.do("POST", "/serving-endpoints/mas-9833b08b-endpoint/invocations",
    body={"input": [{"role": "user",
        "content": "What is the average gold per minute benchmark for Gold ADC (BOTTOM)?"}]})
texts = [c["text"] for it in resp["output"] if it.get("type") == "message"
         for c in it.get("content", []) if c.get("type") == "output_text"]
print(texts[-1])


The average gold per minute benchmark for Gold tier ADC (BOTTOM position) is **466.89**.

This means that Gold-ranked players in the bottom lane typically earn approximately 467 gold per minute on average.


## 4. Coaching narrative (Foundation Model API)
The coach's narrative comes from the Databricks FMAPI (Claude). Real model output:


In [1]:
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
resp = w.serving_endpoints.query(name="databricks-claude-sonnet-4-6", max_tokens=120,
    messages=[
        ChatMessage(role=ChatMessageRole.SYSTEM, content="You are a concise League of Legends coach."),
        ChatMessage(role=ChatMessageRole.USER, content="A Gold ADC has 41 CS at 10 minutes "
            "vs the Gold benchmark of 46. In one sentence, what should they drill?")])
print(resp.choices[0].message.content)


They should drill last-hitting under no pressure (custom game, no enemies) to build the muscle memory of timing auto-attacks to the minion's last hit, since missing CS in lane is almost always a mechanics issue before it's a positioning or wave management issue.


## 5. Lakebase (Postgres) — online benchmark serving + per-user state
The gold benchmarks are synced to a Lakebase Postgres online store (the app reads
them there with low latency, falling back to the warehouse). The same 466.89 value
is served from Postgres, and the app's per-user OLTP tables exist. Connection
mirrors `app/lakebase.py` (mints a fresh Postgres token via the SDK).


In [1]:
import uuid, psycopg2
inst = "league-ai-coach-db"
cred = w.database.generate_database_credential(request_id=str(uuid.uuid4()), instance_names=[inst])
ep = w.database.get_database_instance(name=inst)
conn = psycopg2.connect(host=ep.read_write_dns, port=5432, dbname="databricks_postgres",
    user=w.current_user.me().user_name, password=cred.token, sslmode="require")
cur = conn.cursor()
cur.execute("SELECT tier, team_position, metric, round(gold_avg::numeric,2) "
            "FROM league_ai_coach.gold_challenge_benchmarks_synced "
            "WHERE tier='GOLD' AND team_position='BOTTOM' AND metric='goldPerMinute'")
print("Lakebase online store:", cur.fetchone())
cur.execute("SELECT table_name FROM information_schema.tables WHERE table_schema='app_state' ORDER BY 1")
print("Per-user OLTP tables: ", [r[0] for r in cur.fetchall()])


Lakebase online store: ('GOLD', 'BOTTOM', 'goldPerMinute', Decimal('466.89'))
Per-user OLTP tables:  ['api_usage', 'app_config', 'job_runs', 'llm_cache', 'saved_players', 'search_history', 'user_prefs']


---
**Result:** the app is RUNNING; the Spark Declarative Pipeline populated the
benchmark medallion; the Multi-Agent Supervisor + Genie agents return correct
live answers; the Foundation Model API returns real coaching narratives; and
Lakebase serves the benchmarks + per-user state. End to end, the deployed system works.
